In [3]:
import os
import json
import numpy as np
import datetime
import certifi
import pandas as pd
import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [4]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

Running SQL Alchemy Version: 2.0.34
Running PyMongo Version: 4.16.0


#### Declare & Assign Connection Variables for the MongoDB Server, the MySQL Server & Databases with which You'll be Working 

In [5]:
mysql_args = {
    "uid" : "root",
    "pwd" : "Rf122137182",
    "hostname" : "localhost",  #"wna8fw-mysql.mysql.database.azure.com",
    "dbname" : "adventureworks"
}

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args = {
    "user_name" : "tyd4gf",
    "password" : "Rf122137182",
    "cluster_name" : "lab3",
    "cluster_subnet" : "2g1tl9k",
    "cluster_location" : "atlas", 
    "db_name" : "adventureworks"
}

try:
    engine_stmt = f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}/{mysql_args['dbname']}"
    mysql_engine = create_engine(engine_stmt)

except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to AdventureWorks MySQL database via SQLAlchemy!


In [24]:
customer_query = "SELECT * FROM dim_customers_vw;"

# SQL query into a Pandas dataframe using the SQLAlchemy engine
with mysql_engine.connect() as conn:
    df_customers_staging = pd.read_sql(text(customer_query), conn)

display(df_customers_staging.head(3))

# export the dataframe to a JSON file
json_filename = 'adventureworks_customers.json'
df_customers_staging.to_json(json_filename, orient='records', lines=False)

print(f"\nSuccess! Data exported to {json_filename} in your current directory.")
print(f"Total customer records exported: {len(df_customers_staging)}")

,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,None,Seattle,WA,Washington,b'\x00',98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,None,Renton,WA,Washington,b'\x00',98055,US,United States,North America,Northwest
2,2,AW00000002,S,Main Office,3207 S Grady Way,None,Renton,WA,Washington,b'\x00',98055,US,United States,North America,Northwest



Success! Data exported to adventureworks_customers.json in your current directory.
Total customer records exported: 19220


#### Define Functions for Getting Data From and Setting Data Into Databases

In [7]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

#### Populate MongoDB with Source Data
You only need to run this cell once; however, the operation is *idempotent*.  In other words, it can be run multiple times without changing the end result.

In [25]:
client = get_mongo_client(**mongodb_args)

data_dir = os.getcwd()

json_files = {
    "customers" : "adventureworks_customers.json"
}

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)

### 1.0. Extract Source Data for Dimension Tables
In a dimensional model, dimension tables provide the "who, what, where, and when" context for our business process. In this section, we extract data from three different source systems to build our dimensions: a NoSQL document database (MongoDB), a relational database (MySQL), and a local file system (CSV).

#### 1.1. Extract Customer Data from MongoDB
This cell connects to our cloud-based MongoDB Atlas cluster to extract the customer demographic data that we previously exported from AdventureWorks.

In [26]:
client = get_mongo_client(**mongodb_args)

query = {} # Select all elements (columns), and all documents (rows).
collection = "customers"

df_dim_customers = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
df_dim_customers.head(2)

,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,None,Seattle,WA,Washington, ,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,None,Renton,WA,Washington, ,98055,US,United States,North America,Northwest


#### 1.2 Creating CSV File to Extract From
Every data warehouse requires a Date Dimension to perform time-series analysis. This cell generates a CSV file of dates and immediately reads it into a Pandas DataFrame, fulfilling the requirement to extract data from a file system.

In [30]:
# 1. date dimension CSV file 
dates = pd.date_range(start='2001-01-01', end='2005-12-31')
df_date_gen = pd.DataFrame({
    'DateKey': dates.strftime('%Y%m%d').astype(int),
    'FullDate': dates,
    'Year': dates.year,
    'Month': dates.month,
    'DayOfWeek': dates.day_name()
})
df_date_gen.to_csv('dim_date.csv', index=False)
print("dim_date.csv created successfully on your local drive!")

# 2. extracting the data locally
df_dim_date = pd.read_csv('dim_date.csv')
print(f"Successfully extracted {len(df_dim_date)} Dates from CSV!")
display(df_dim_date.head(2))

dim_date.csv created successfully on your local drive!
Successfully extracted 1826 Dates from CSV!


,DateKey,FullDate,Year,Month,DayOfWeek
0,20010101,2001-01-01,2001,1,Monday
1,20010102,2001-01-02,2001,1,Tuesday


#### 1.3. Extract Product Data from MySQL
These cells connect to the AdventureWorks MySQL database to extract data.

In [31]:
# Extract the Products Dimension from MySQL
sql_dim_products = "SELECT * FROM dim_products_vw;"
df_dim_products = get_sql_dataframe(sql_dim_products, **mysql_args)
df_dim_products.head(2)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,DaysToManufacture,ProductLine,Class,Style,ProductCategory,ProductSubcategory,ProductModel,SellStartDate,SellEndDate,DiscontinuedDate
0,1,Adjustable Race,AR-5381,b'\x00',b'\x00',None,1000,750,$0.00,$0.00,...,0,None,None,None,None,None,None,1998-06-01,NaT,None
1,2,Bearing Ball,BA-8327,b'\x00',b'\x00',None,1000,750,$0.00,$0.00,...,0,None,None,None,None,None,None,1998-06-01,NaT,None


### 2.0. Transform and Load Dimension Tables
#### 2.1. Transform Dimension DataFrames
Before loading data into our destination Data Mart, we must standardize the column names. We rename the source system IDs (Business Keys) to ensure they don't clash with the auto-incrementing Surrogate Keys our pipeline will generate in the next step.

In [33]:
df_dim_customers.rename(columns={"CustomerID": "customer_id"}, inplace=True)
df_dim_customers.drop_duplicates(subset=['customer_id'], inplace=True)
# keep only relevant analytical columns, drop N/A heavy columns 
cust_columns = ['customer_id', 'City', 'State_Province', 'Country_Region', 'PostalCode']
df_dim_customers = df_dim_customers[cust_columns]

df_dim_products.rename(columns={"ProductID": "product_id"}, inplace=True)
# keep core product details, drop operational flags and mostly-null columns
prod_columns = ['product_id', 'Name', 'ProductNumber', 'StandardCost', 'ListPrice']
df_dim_products = df_dim_products[prod_columns]

df_dim_date.rename(columns={"date_key": "date_id"}, inplace=True)

#### 2.2. Initialize the Target Data Warehouse
Here we create a clean, dedicated MySQL database to house our final Star Schema data mart, ensuring we do not overwrite our transactional source data.

In [52]:
with mysql_engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS adventureworks_dw;"))
    
mysql_dw_args = mysql_args.copy()
mysql_dw_args["dbname"] = "adventureworks_dw"

#### 2.3. Load Dimensions into the Data Warehouse
Using the `set_dataframe` function, we push our cleaned DataFrames into the new data mart. This function automatically creates the tables and appends an auto-incrementing integer primary key (Surrogate Key) to each table.

In [53]:
set_dataframe(df_dim_customers, 'dim_customers', 'customer_key', 'insert', **mysql_dw_args)
# print("dim_customers load please!")

# Load Products
set_dataframe(df_dim_products, 'dim_products', 'product_key', 'insert', **mysql_dw_args)
# print("dim_products loadd!")

# Load Dates
set_dataframe(df_dim_date, 'dim_date', 'date_key', 'insert', **mysql_dw_args)
# print("dim_date loaddd plz !")

In [54]:
sql_validate_customers = "SELECT customer_key, customer_id, City FROM adventureworks_dw.dim_customers;"
df_val_cust = get_sql_dataframe(sql_validate_customers, **mysql_dw_args)
df_val_cust.head(2)

,customer_key,customer_id,City
0,1,1,Seattle
1,2,2,Renton


### 3.0. Create and Populate the Fact Table
The Fact table sits at the center of the Star Schema, recording the quantitative measurements of the business process. 

In [55]:
# Extract the Fact Sales Orders from MySQL
sql_fact_sales = "SELECT * FROM fact_sales_orders_vw;"
df_fact_sales = get_sql_dataframe(sql_fact_sales, **mysql_args)
df_fact_sales.head(2)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardApprovalCode,SubTotal,TaxAmt,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,105041Vi84182,"$24,643.94","$1,971.51",$616.10,"$27,231.55",4911-403C-98,4,711,$20.19,$80.75
1,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,105041Vi84182,"$24,643.94","$1,971.51",$616.10,"$27,231.55",4911-403C-98,2,712,$5.19,$10.37


#### 3.1. Fetch Generated Surrogate Keys
To properly link our Fact table to our Dimension tables, we must retrieve the newly generated Surrogate Keys (`customer_key`, `product_key`, `date_key`) from our data mart.

In [56]:
# Fetch the newly generated Surrogate Keys from the Data Warehouse
sql_dim_cust = "SELECT customer_key, customer_id FROM dim_customers;"
df_dim_cust_keys = get_sql_dataframe(sql_dim_cust, **mysql_dw_args)

sql_dim_prod = "SELECT product_key, product_id FROM dim_products;"
df_dim_prod_keys = get_sql_dataframe(sql_dim_prod, **mysql_dw_args)

sql_dim_date = "SELECT date_key, FullDate FROM dim_date;"
df_dim_date_keys = get_sql_dataframe(sql_dim_date, **mysql_dw_args)

df_fact_sales['OrderDate'] = pd.to_datetime(df_fact_sales['OrderDate'])
df_dim_date_keys['FullDate'] = pd.to_datetime(df_dim_date_keys['FullDate'])

# print("Surrogate keys is this running!")

#### 3.2. Merge Surrogate Keys into the Fact DataFrame
We perform inner joins using Pandas to map the original Business Keys in our Fact table to the new Surrogate Keys.

In [57]:
# Rename fact table columns
df_fact_sales.rename(columns={'CustomerID': 'customer_id', 'ProductID': 'product_id'}, inplace=True)

# Merge Customers
df_fact_sales = pd.merge(df_fact_sales, df_dim_cust_keys, on='customer_id', how='inner')

# Merge Products
df_fact_sales = pd.merge(df_fact_sales, df_dim_prod_keys, on='product_id', how='inner')

# FIXED: Merge using right_on='FullDate'
df_fact_sales = pd.merge(df_fact_sales, df_dim_date_keys, left_on='OrderDate', right_on='FullDate', how='inner')

print(f"Merges complete plz! Current row count: {len(df_fact_sales)}")

Merges complete plz! Current row count: 121317


#### 3.3. Clean and Load the Final Fact Table
A Fact table should only contain metrics and foreign keys. We drop all unnecessary descriptive columns and push the final, flattened DataFrame into the data mart.

In [58]:
final_columns = [
    'SalesOrderID', 'customer_key', 'product_key', 'date_key',
    'OrderQty', 'UnitPrice', 'LineTotal', 'TaxAmt', 'Freight'
]
df_fact_sales_final = df_fact_sales[final_columns].copy()

set_dataframe(df_fact_sales_final, 'fact_sales', 'fact_sales_key', 'insert', **mysql_dw_args)

### 4.0. Validate the Star Schema Architecture
To prove our ETL pipeline was successful, we execute a SQL query against our new Data Mart. By joining the central Fact table out to the Dimensions, we can easily aggregate metrics (like Total Revenue) across descriptive categories (like Product Name and Day of the Week).

In [64]:
pd.options.display.float_format = '${:,.2f}'.format # this is to make numbers format better

# total revenue and quantity sold by year
sql_validation_1 = """
SELECT 
    d.Year,
    SUM(f.OrderQty) AS total_quantity,
    SUM(f.LineTotal) AS total_revenue
FROM adventureworks_dw.fact_sales f
JOIN adventureworks_dw.dim_date d ON f.date_key = d.date_key
GROUP BY d.Year
ORDER BY d.Year;
"""

df_val_1 = get_sql_dataframe(sql_validation_1, **mysql_dw_args)
df_val_1.head(10)

,Year,total_quantity,total_revenue
0,2001,"$11,848.00","$11,331,808.96"
1,2002,"$60,918.00","$30,674,773.17"
2,2003,"$124,699.00","$42,011,037.16"
3,2004,"$77,449.00","$25,828,762.10"


#### 4.1. Additional Validation: Top 10 Products by Total Revenue
This query joins the Fact table to the Products dimension to calculate the highest-grossing items in the catalog.

In [65]:
# top 10 products by total revenue
sql_validation_2 = """
SELECT 
    p.Name AS product_name,
    SUM(f.OrderQty) AS total_quantity,
    SUM(f.LineTotal) AS total_revenue
FROM adventureworks_dw.fact_sales f
JOIN adventureworks_dw.dim_products p ON f.product_key = p.product_key
GROUP BY p.Name
ORDER BY total_revenue DESC;
"""

df_val_2 = get_sql_dataframe(sql_validation_2, **mysql_dw_args)
df_val_2.head(10)

,product_name,total_quantity,total_revenue
0,"Mountain-200 Black, 38","$2,977.00","$4,400,592.80"
1,"Mountain-200 Black, 42","$2,664.00","$4,009,494.76"
2,"Mountain-200 Silver, 38","$2,394.00","$3,693,678.03"
3,"Mountain-200 Silver, 42","$2,234.00","$3,438,478.86"
4,"Mountain-200 Silver, 46","$2,216.00","$3,434,256.94"
5,"Mountain-200 Black, 46","$2,111.00","$3,309,673.22"
6,"Road-250 Black, 44","$1,642.00","$2,516,857.31"
7,"Road-250 Black, 48","$1,498.00","$2,347,655.95"
8,"Road-250 Black, 52","$1,245.00","$2,012,447.78"
9,"Road-150 Red, 56",$664.00,"$1,847,818.63"


#### 4.2. Additional Validation: Revenue by Customer City and Year
To further prove the robustness of our Star Schema, this query joins the Fact table to both the Customer and Date dimensions. It aggregates the total number of distinct orders and total revenue, grouped by the customer's city and the year the order was placed.

In [66]:
# revenue by customer city and year
sql_validation_3 = """
SELECT 
    c.City,
    d.Year,
    COUNT(DISTINCT f.SalesOrderID) AS TotalOrders,
    SUM(f.LineTotal) AS TotalRevenue
FROM adventureworks_dw.fact_sales f
JOIN adventureworks_dw.dim_customers c ON f.customer_key = c.customer_key
JOIN adventureworks_dw.dim_date d ON f.date_key = d.date_key
GROUP BY c.City, d.Year
ORDER BY TotalRevenue DESC;
"""

df_val_3 = get_sql_dataframe(sql_validation_3, **mysql_dw_args)
df_val_3.head(10)

,City,Year,TotalOrders,TotalRevenue
0,Toronto,2003,61,"$1,620,467.96"
1,Toronto,2002,57,"$1,592,322.59"
2,London,2003,327,"$1,292,292.31"
3,Paris,2003,249,"$972,535.03"
4,London,2004,336,"$803,348.38"
5,Toronto,2004,32,"$666,771.26"
6,Toronto,2001,26,"$580,915.88"
7,London,2002,72,"$579,857.39"
8,Seattle,2003,49,"$566,760.54"
9,Richmond,2003,17,"$560,108.20"
